# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")
print(f"Published: {getattr(metadata, 'datePublished', None)}")
print(f"Identifier: {getattr(metadata, 'identifier', None)}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# List the available record sets and their fields, referencing by @id

record_sets = getattr(metadata, 'recordSet', [])
if not record_sets:
    print('No record sets found in the metadata.')
else:
    for rs in record_sets:
        print(f"RecordSet @id: {rs['@id']}")
        fields = rs.get('field', [])
        if isinstance(fields, dict):
            fields = [fields]
        field_ids = [f['@id'] for f in fields]
        print(f"  Fields (@id): {field_ids}")

In [ ]:
# If present, show a preview of records for each RecordSet
if not record_sets:
    print('No record sets available for record preview.')
else:
    # Example: show for the first record set
    first_record_set_id = record_sets[0]['@id']
    print(f"Displaying first 2 records from RecordSet: {first_record_set_id}")
    for i, record in enumerate(dataset.records(record_set=first_record_set_id)):
        print(record)
        if i >= 1:
            break

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Collect and load each record set into a pandas DataFrame
dataframes = {}
record_set_ids = []

if not record_sets:
    print('No record sets found in the metadata.')
else:
    for rs in record_sets:
        rs_id = rs['@id']
        print(f"Loading records for RecordSet @id: {rs_id}")
        record_set_ids.append(rs_id)
        records = list(dataset.records(record_set=rs_id))
        df = pd.DataFrame(records)
        dataframes[rs_id] = df
        print(f"Columns for {rs_id}: {df.columns.tolist()}")

# Preview the first record set's DataFrame
if record_set_ids:
    preview_id = record_set_ids[0]
    print(f"First 5 rows for {preview_id}:")
    display(dataframes[preview_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section includes operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

In [ ]:
import numpy as np

# Example: Select a numeric field for analysis from the available columns
if record_set_ids:
    rs_id = record_set_ids[0]
    df = dataframes[rs_id]
    # Find likely numeric columns (float or int)
    numeric_fields = [c for c in df.columns if pd.api.types.is_numeric_dtype(df[c])]
    if not numeric_fields:
        # Try to coerce any likely numeric columns
        for c in df.columns:
            try:
                df[c] = pd.to_numeric(df[c])
            except Exception:
                continue
        numeric_fields = [c for c in df.columns if pd.api.types.is_numeric_dtype(df[c])]
    if numeric_fields:
        numeric_field = numeric_fields[0]
        print(f"Analyzing numeric field (by @id/column name): {numeric_field}")
        threshold = df[numeric_field].quantile(0.75)  # Use 75th percentile for demonstration
        filtered_df = df[df[numeric_field] > threshold].copy()
        print(f"Filtered records with {numeric_field} > {threshold}")
        display(filtered_df.head())
        # Normalize
        filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(f"Normalized {numeric_field} for filtered records:")
        display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

        # Attempt grouping by a non-numeric field (if any present)
        group_fields = [c for c in df.columns if c != numeric_field and not pd.api.types.is_numeric_dtype(df[c])]
        if group_fields:
            group_field = group_fields[0]
            print(f"Grouping records by field: {group_field}")
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
            display(grouped_df.head())
        else:
            print('No categorical field found for grouping.')
    else:
        print('No numeric fields found for EDA.')
else:
    print('No record sets loaded for analysis.')

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Plot distributions for numeric field in the first record set
if record_set_ids and numeric_fields:
    rs_id = record_set_ids[0]
    df = dataframes[rs_id]
    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_field].dropna(), bins=30, kde=True)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.tight_layout()
    plt.show()
    # If group_field exists, plot boxplot
    if 'group_field' in locals():
        plt.figure(figsize=(10, 6))
        sns.boxplot(x=group_field, y=numeric_field, data=df)
        plt.xticks(rotation=45)
        plt.title(f"{numeric_field} by {group_field}")
        plt.tight_layout()
        plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

In this notebook, we loaded FAIR^2 mass spectrometry data from extracellular vesicles isolated from human mast cells using the `mlcroissant` library, directly referencing all resources and fields by their Croissant `@id`.
- We reviewed the dataset's available record sets and fields metadata.
- We extracted records into pandas DataFrames for inspection.
- Basic EDA illustrated data selection, normalization, grouping, and visualization for numeric and categorical attributes.

You can continue further with advanced analyses by adjusting the selected `@id` fields as needed for your research!